In [ ]:
# PCAP Flow-Purity Audit for the 10-App Mobile Dataset
# Google Colab-ready
#
# INPUT:
#   /content/drive/MyDrive/mobile
#
# OUTPUT:
#   /content/drive/MyDrive/mobile_purity_audit
#
# This script AUDITS only. It does not modify or delete original PCAP files.
#
# Categories:
#   Target
#   Other-app/background
#   System/background
#   Third-party/shared
#   Unresolved

# ---- Colab setup ----
from google.colab import drive
drive.mount("/content/drive")

!apt-get -qq update
!apt-get -qq install -y tshark

import os
import re
import glob
import ipaddress
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

INPUT_DIR = Path("/content/drive/MyDrive/mobile")
OUTPUT_DIR = Path("/content/drive/MyDrive/mobile_purity_audit_v2")
DETAIL_DIR = OUTPUT_DIR / "per_capture"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DETAIL_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Application attribution rules ----------
# Keep rules conservative. These are evidence for attribution, not ML features.
APP_TERMS = {
    "facebook": [
        "facebook.com", "fbcdn.net", "fbsbx.com", "facebook.net"
    ],
    "instagram": [
        "instagram.com", "cdninstagram.com"
    ],
    "linkedin": [
        "linkedin.com", "licdn.com"
    ],
    "reddit": [
        "reddit.com", "redd.it", "redditmedia.com", "redditstatic.com"
    ],
    "snapchat": [
        "snapchat.com", "sc-cdn.net", "sc-gw.com", "snapkit.com"
    ],
    "telegram": [
        "telegram.org", "t.me"
    ],
    "tiktok": [
        "tiktok.com", "tiktokv.com", "tiktokv.us", "tiktokcdn.com",
        "tiktokcdn-us.com", "ttoverseaus.net", "bytedance.com",
        "byteoversea.com", "ibytedtos.com", "byteimg.com",
        "muscdn.com", "musical.ly"
    ],
    "twitter": [
        "twitter.com", "x.com", "twimg.com", "t.co"
    ],
    "whatsapp": [
        "whatsapp.com", "whatsapp.net"
    ],
    "youtube": [
        "youtube.com", "youtu.be", "googlevideo.com", "ytimg.com",
        "youtubei.googleapis.com"
    ],
}

# Services that may support many apps. Do NOT automatically call these contamination.
SHARED_TERMS = [
    "googleapis.com", "google.com", "gstatic.com",
    "firebase.google.com", "firebaseio.com", "app-measurement.com",
    "doubleclick.net", "googletagmanager.com",
    "cloudfront.net", "amazonaws.com",
    "akamaiedge.net", "akamaihd.net", "edgesuite.net",
    "fastly.net",
    "branch.io", "appsflyer",
    "clarity.ms", "bing.com", "microsoft.com",
]

# Telegram infrastructure: conservative known Telegram network ranges.
TELEGRAM_NETS = [
    ipaddress.ip_network("149.154.160.0/20"),
    ipaddress.ip_network("91.108.4.0/22"),
    ipaddress.ip_network("91.108.8.0/22"),
    ipaddress.ip_network("91.108.12.0/22"),
    ipaddress.ip_network("91.108.16.0/22"),
    ipaddress.ip_network("91.108.20.0/22"),
    ipaddress.ip_network("91.108.56.0/22"),
]

def norm_app_name(filename):
    """Infer intended app only from the controlled capture filename."""
    name = Path(filename).stem.lower()
    aliases = {"x": "twitter"}
    for app in APP_TERMS:
        if re.search(rf"(^|[_\-\s]){re.escape(app)}([_\-\s]|$)", name):
            return app
    for alias, app in aliases.items():
        if re.search(rf"(^|[_\-\s]){re.escape(alias)}([_\-\s]|$)", name):
            return app
    return None

def in_telegram_net(ip):
    try:
        addr = ipaddress.ip_address(str(ip))
        return any(addr in net for net in TELEGRAM_NETS)
    except Exception:
        return False

def is_local_or_system(ip, port):
    try:
        addr = ipaddress.ip_address(str(ip))
        if addr.is_multicast or addr.is_unspecified:
            return True
        if str(ip) == "255.255.255.255":
            return True
    except Exception:
        pass
    try:
        port = int(float(port))
    except Exception:
        port = -1
    return port in {67, 68, 1900, 5353, 3702}

def evidence_hostnames(row):
    """
    Return normalized hostnames observed in TLS SNI and DNS query fields.
    Matching is performed on complete DNS labels, not arbitrary substrings.
    """
    raw_values = [
        row.get("tls_sni", ""),
        row.get("dns_names", ""),
    ]

    hosts = set()
    for raw in raw_values:
        if pd.isna(raw):
            continue
        # tshark may return multiple values separated by commas or semicolons.
        for value in re.split(r"[,;\s]+", str(raw).lower()):
            host = value.strip().strip(".")
            if not host:
                continue
            # Remove a possible port from a hostname, but preserve IPv6 values.
            if host.count(":") == 1:
                host = host.split(":", 1)[0]
            hosts.add(host)
    return hosts

def domain_matches(host, domain):
    """
    Exact domain/subdomain match.

    Examples:
      x.com matches x.com and api.x.com
      t.co matches t.co and api.t.co
      t.co DOES NOT match yt3.ggpht.com
    """
    host = str(host).lower().strip().strip(".")
    domain = str(domain).lower().strip().strip(".")
    return host == domain or host.endswith("." + domain)

def any_domain_match(hosts, domains):
    return any(domain_matches(host, domain) for host in hosts for domain in domains)

def classify_flow(row, target_app):
    hosts = evidence_hostnames(row)

    # 1. Direct target-domain/infrastructure evidence.
    if any_domain_match(hosts, APP_TERMS[target_app]):
        return "Target", "target domain/infrastructure"

    # 2. Telegram dedicated address space.
    if target_app == "telegram":
        if in_telegram_net(row.get("src_ip", "")) or in_telegram_net(row.get("dst_ip", "")):
            return "Target", "Telegram IP range"

    # 3. Local/system discovery traffic.
    if is_local_or_system(row.get("dst_ip", ""), row.get("dst_port", "")):
        return "System/background", "local broadcast/multicast/system service"

    # 4. Strong evidence of another named application.
    other_hits = []
    for app, domains in APP_TERMS.items():
        if app == target_app:
            continue
        if any_domain_match(hosts, domains):
            other_hits.append(app)

    if other_hits:
        return (
            "Other-app/background",
            "observable evidence: " + ",".join(sorted(set(other_hits)))
        )

    # 5. Shared services are NOT treated as contamination.
    if any_domain_match(hosts, SHARED_TERMS):
        return "Third-party/shared", "shared CDN/analytics/platform infrastructure"

    # 6. Insufficient evidence: retain as unresolved rather than guessing.
    return "Unresolved", "insufficient attribution evidence"

# ---------- tshark extraction ----------
FIELDS = [
    "frame.number",
    "frame.time_epoch",
    "ip.src",
    "ipv6.src",
    "ip.dst",
    "ipv6.dst",
    "tcp.srcport",
    "udp.srcport",
    "tcp.dstport",
    "udp.dstport",
    "frame.len",
    "tls.handshake.extensions_server_name",
    "dns.qry.name",
]

def run_tshark(pcap):
    cmd = [
        "tshark", "-r", str(pcap),
        "-Y", "ip || ipv6",
        "-T", "fields",
        "-E", "header=y",
        "-E", "separator=\t",
        "-E", "quote=d",
        "-E", "occurrence=a",
    ]
    for field in FIELDS:
        cmd += ["-e", field]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-2000:])

    from io import StringIO
    if not result.stdout.strip():
        return pd.DataFrame()
    return pd.read_csv(StringIO(result.stdout), sep="\t", dtype=str).fillna("")

def first_nonempty(*values):
    for value in values:
        if value is not None and str(value).strip():
            return str(value).strip()
    return ""

def packets_to_flows(pkt):
    """
    Construct bidirectional 5-tuple flows from packet metadata.
    DNS names and TLS SNI observed on packets are propagated to the flow.
    This is an audit representation, not the NFStream training dataset.
    """
    rows = []

    # DNS response mapping learned within the capture.
    # tshark extraction above records query names but not A/AAAA answers,
    # so names are used as observable evidence on the same flow only.
    for _, r in pkt.iterrows():
        src_ip = first_nonempty(r.get("ip.src"), r.get("ipv6.src"))
        dst_ip = first_nonempty(r.get("ip.dst"), r.get("ipv6.dst"))
        src_port = first_nonempty(r.get("tcp.srcport"), r.get("udp.srcport"))
        dst_port = first_nonempty(r.get("tcp.dstport"), r.get("udp.dstport"))

        proto = "tcp" if first_nonempty(r.get("tcp.srcport")) else "udp"
        if not src_ip or not dst_ip:
            continue

        try:
            sp = int(float(src_port)) if src_port else 0
            dp = int(float(dst_port)) if dst_port else 0
        except Exception:
            sp, dp = 0, 0

        # Canonical bidirectional key.
        a = (src_ip, sp)
        b = (dst_ip, dp)
        if a <= b:
            key = (proto, a[0], a[1], b[0], b[1])
        else:
            key = (proto, b[0], b[1], a[0], a[1])

        rows.append({
            "key": key,
            "time": pd.to_numeric(r.get("frame.time_epoch", ""), errors="coerce"),
            "bytes": pd.to_numeric(r.get("frame.len", ""), errors="coerce"),
            "tls_sni": str(r.get("tls.handshake.extensions_server_name", "")),
            "dns_names": str(r.get("dns.qry.name", "")),
        })

    if not rows:
        return pd.DataFrame()

    temp = pd.DataFrame(rows)

    def unique_text(series):
        values = []
        for item in series:
            for v in str(item).split(","):
                v = v.strip()
                if v and v not in values:
                    values.append(v)
        return ";".join(values)

    grouped = []
    for key, g in temp.groupby("key", sort=False):
        proto, ip1, port1, ip2, port2 = key
        grouped.append({
            "protocol": proto,
            "src_ip": ip1,
            "src_port": port1,
            "dst_ip": ip2,
            "dst_port": port2,
            "first_seen": g["time"].min(),
            "last_seen": g["time"].max(),
            "duration_s": g["time"].max() - g["time"].min(),
            "packets": len(g),
            "bytes": g["bytes"].fillna(0).sum(),
            "tls_sni": unique_text(g["tls_sni"]),
            "dns_names": unique_text(g["dns_names"]),
        })
    return pd.DataFrame(grouped)


# ---------- Audit all captures ----------
pcaps = sorted(
    list(INPUT_DIR.rglob("*.pcap")) +
    list(INPUT_DIR.rglob("*.pcapng"))
)

print(f"Found {len(pcaps)} PCAP/PCAPNG files in {INPUT_DIR}")

summary_rows = []
errors = []

for i, pcap in enumerate(pcaps, 1):
    app = norm_app_name(pcap.name)

    if app is None:
        print(f"[{i}/{len(pcaps)}] SKIP: cannot infer app from {pcap.name}")
        errors.append({"capture_file": pcap.name, "error": "could not infer target app"})
        continue

    print(f"[{i}/{len(pcaps)}] Auditing {pcap.name} -> {app}")

    try:
        packets = run_tshark(pcap)
        flows = packets_to_flows(packets)

        if flows.empty:
            errors.append({"capture_file": pcap.name, "error": "no IP flows extracted"})
            continue

        result = flows.apply(lambda r: classify_flow(r, app), axis=1)
        flows["target_app"] = app
        flows["category"] = [x[0] for x in result]
        flows["reason"] = [x[1] for x in result]
        flows["capture_file"] = pcap.name

        # Preserve relative folder information if the 50 files are organized in subfolders.
        rel = pcap.relative_to(INPUT_DIR)
        safe_rel = "__".join(rel.parts)
        safe_rel = re.sub(r"\.(pcapng|pcap)$", "", safe_rel, flags=re.I)
        detail_path = DETAIL_DIR / f"{safe_rel}_purity_audit.csv"
        flows.to_csv(detail_path, index=False)

        total_flows = len(flows)
        total_packets = flows["packets"].sum()
        total_bytes = flows["bytes"].sum()

        for category, g in flows.groupby("category"):
            summary_rows.append({
                "capture_file": pcap.name,
                "relative_path": str(rel),
                "target_app": app,
                "category": category,
                "flows": len(g),
                "flow_pct": 100 * len(g) / total_flows,
                "packets": int(g["packets"].sum()),
                "packet_pct": 100 * g["packets"].sum() / total_packets if total_packets else np.nan,
                "bytes": int(g["bytes"].sum()),
                "byte_pct": 100 * g["bytes"].sum() / total_bytes if total_bytes else np.nan,
            })

    except Exception as e:
        errors.append({"capture_file": pcap.name, "error": str(e)})
        print("  ERROR:", e)

summary = pd.DataFrame(summary_rows)
summary_file = OUTPUT_DIR / "pcap_purity_summary.csv"
summary.to_csv(summary_file, index=False)

if errors:
    pd.DataFrame(errors).to_csv(OUTPUT_DIR / "audit_errors.csv", index=False)

# Capture-level wide report
if not summary.empty:
    wide = summary.pivot_table(
        index=["capture_file", "target_app"],
        columns="category",
        values=["flows", "flow_pct", "packet_pct", "byte_pct"],
        aggfunc="first",
        fill_value=0
    )
    wide.columns = ["__".join(map(str, c)) for c in wide.columns]
    wide = wide.reset_index()
    wide.to_csv(OUTPUT_DIR / "capture_level_purity_report.csv", index=False)

    # App-level aggregate
    app_summary = (
        summary.groupby(["target_app", "category"], as_index=False)
        [["flows", "packets", "bytes"]].sum()
    )
    totals = app_summary.groupby("target_app")[["flows", "packets", "bytes"]].transform("sum")
    app_summary["flow_pct"] = 100 * app_summary["flows"] / totals["flows"]
    app_summary["packet_pct"] = 100 * app_summary["packets"] / totals["packets"]
    app_summary["byte_pct"] = 100 * app_summary["bytes"] / totals["bytes"]
    app_summary.to_csv(OUTPUT_DIR / "application_level_purity_report.csv", index=False)

print("\nAUDIT COMPLETE")
print("Original PCAPs were not modified.")
print("Results:", OUTPUT_DIR)
print("Main files:")
print(" - pcap_purity_summary.csv")
print(" - capture_level_purity_report.csv")
print(" - application_level_purity_report.csv")
print(" - per_capture/*.csv")
if errors:
    print(" - audit_errors.csv")
